# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [6]:
%pip install -qU jq lark langchain langchain-classic langchain-community langchain-elasticsearch langchain-openai langchain-text-splitters tiktoken

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever


Note: you may need to restart the kernel to use updated packages.


## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [5]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Use the exact .env file for this lab.
env_path = Path("/Users/omari/Documents/Academic/Ironhack_AI_Bootcamp/Bootcamp_W7/D4/lab-chatbot-with-multi-query-retriever/.env")
load_dotenv(env_path)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Your .env currently uses these names: CouldID and Elasticsearch_API.
# The fallback names are included in case you rename them later.
ELASTIC_CLOUD_ID = os.getenv("ELASTIC_CLOUD_ID") or os.getenv("CouldID")
ELASTIC_API_KEY = os.getenv("ELASTIC_API_KEY") or os.getenv("Elasticsearch_API")
ELASTIC_INDEX_NAME = "multi-query-chatbot"

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is missing from .env")
if not ELASTIC_CLOUD_ID:
    raise ValueError("Elastic Cloud ID is missing from .env. Use ELASTIC_CLOUD_ID=... or CouldID=...")
if not ELASTIC_API_KEY:
    raise ValueError("Elastic API key is missing from .env. Use ELASTIC_API_KEY=... or Elasticsearch_API=...")

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name=ELASTIC_INDEX_NAME,
    embedding=embeddings,
)


## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [7]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

In [10]:
type(data)
data[0]


{'content': "Effective: March 2020\nPurpose\n\nThe purpose of this full-time work-from-home policy is to provide guidelines and support for employees to conduct their work remotely, ensuring the continuity and productivity of business operations during the COVID-19 pandemic and beyond.\nScope\n\nThis policy applies to all employees who are eligible for remote work as determined by their role and responsibilities. It is designed to allow employees to work from home full time while maintaining the same level of performance and collaboration as they would in the office.\nEligibility\n\nEmployees who can perform their work duties remotely and have received approval from their direct supervisor and the HR department are eligible for this work-from-home arrangement.\nEquipment and Resources\n\nThe necessary equipment and resources will be provided to employees for remote work, including a company-issued laptop, software licenses, and access to secure communication tools. Employees are respon

In [11]:
import pandas as pd

df = pd.DataFrame(data)
df.head()


,content,summary,name,url,created_on,updated_at,category,_run_ml_inference,rolePermissions,restricted
0,Effective: March 2020\nPurpose\n\nThe purpose ...,This policy outlines the guidelines for full-t...,Work From Home Policy,./sharepoint/Work from home policy.txt,2020-03-01,2020-03-01,teams,True,"[demo, manager]",NaN
1,"Starting May 2022, the company will be impleme...","Starting May 2022, employees will need to work...",April Work From Home Update,./sharepoint/April work from home update.txt,2022-04-29,2022-04-29,teams,True,"[demo, manager]",NaN
2,As we continue to prioritize the well-being of...,"Starting May 1, 2023, our hybrid work policy w...",Wfh Policy Update May 2023,./sharepoint/WFH policy update May 2023.txt,2023-05-01,2023-05-01,teams,True,"[demo, manager]",NaN
3,Executive Summary:\nThis sales strategy docume...,This sales strategy document outlines objectiv...,Fy2024 Company Sales Strategy,./sharepoint/FY2024 Company Sales Strategy.txt,2023-04-15,2023-04-15,teams,True,"[demo, manager]",NaN
4,Purpose\n\nThe purpose of this vacation policy...,: This policy outlines the guidelines and proc...,Company Vacation Policy,https://enterprisesearch.sharepoint.com/:t:/s/...,2018-04-15,2018-04-16,sharepoint,True,"[demo, manager]",NaN


### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [12]:
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    # Add useful fields from each JSON record into document metadata
    metadata["name"] = record.get("name")
    metadata["summary"] = record.get("summary")
    metadata["url"] = record.get("url")
    metadata["category"] = record.get("category")
    metadata["updated_at"] = record.get("updated_at")

    return metadata


loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800,
    chunk_overlap=400
)

docs = loader.load_and_split(text_splitter=text_splitter)


### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [13]:
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name=ELASTIC_INDEX_NAME,
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [14]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.prompts import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

---- Answer ----
The NASA sales team is a part of the Americas region in the sales organization of the company. It is led by two Area Vice-Presidents, Laura Martinez for North America and Gary Johnson for South America. The team is responsible for promoting and selling the company's products and services in the North and South American markets. They work closely with other departments, such as marketing, product development, and customer support, to ensure the company's success in these regions.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [15]:
question = "what is the nasa sales team?"

normal_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
normal_docs = normal_retriever.invoke(question)

multi_docs = retriever.invoke(question)

print("NORMAL RETRIEVER RESULTS")
for doc in normal_docs:
    print(doc.metadata.get("name"))
    print(doc.page_content[:300])
    print("-" * 50)

print("\nMULTI-QUERY RETRIEVER RESULTS")
for doc in multi_docs:
    print(doc.metadata.get("name"))
    print(doc.page_content[:300])
    print("-" * 50)


NORMAL RETRIEVER RESULTS
Sales Organization Overview
Our sales organization is structured to effectively serve our customers and achieve our business objectives across multiple regions. The organization is divided into the following main regions:

The Americas: This region includes the United States, Canada, Mexico, as well as Central and South Americ
--------------------------------------------------
Sales Engineering Collaboration
Title: Working with the Sales Team as an Engineer in a Tech Company

Introduction:
As an engineer in a tech company, collaboration with the sales team is essential to ensure the success of the company's products and services. This guidance document aims to provide an overview of how engineers can ef
--------------------------------------------------
Fy2024 Company Sales Strategy
Executive Summary:
This sales strategy document outlines the key objectives, focus areas, and action plans for our tech company's sales operations in fiscal year 2024. Our primary g